# 03 Final Training And Hidden Threshold

Load the best Optuna family, freeze the non-hidden hyperparameters, run a cached low-dimensional hidden-size sweep on CPU, compare it against the existing `h=4` reference run, and optionally retrain the final chosen model.

In [ ]:
from pathlib import Path

ARTIFACT_ROOT = Path('artifacts')
TRAIN_PATH = Path('Coursework3/viscodata_3mat.mat')
DEVICE = 'cpu'
SEED = 20260328
SWEEP_SEEDS = [20260328, 20260329, 20260330]
HIDDEN_GRID = [0, 1, 2, 3]
REFERENCE_RESULTS_PATH = ARTIFACT_ROOT / 'final' / 'hidden_threshold_adaptive_results.csv'
REFERENCE_HIDDEN = 4
TOLERANCE_RATIO = 0.05
RUN_PREFIX = 'hidden_threshold_low_dim'
HIDDEN_VERBOSE = True
RUN_FINAL_RETRAIN = False
FINAL_RUN_NAME = 'final_model'


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import Image, display

from Coursework3.RNO_1D_Skeleton import (
    ExperimentConfig,
    pick_best_family,
    plot_hidden_threshold,
    prepare_data,
    retrain_final_model,
    run_hidden_threshold_grid_progressive,
    write_json,
)

os.environ['COURSEWORK_DEVICE'] = DEVICE

data_bundle = prepare_data(train_path=TRAIN_PATH, artifact_root=ARTIFACT_ROOT, split_seed=SEED)
best_param_paths = sorted(Path(ARTIFACT_ROOT / 'optuna').glob('best_params_*.json'))
core_type, best_payload, best_path = pick_best_family(best_param_paths)
base_config = ExperimentConfig(**json.loads(best_payload['best_user_attrs']['config']))
base_config.CORE_TYPE = core_type
base_config.MODEL_TAG = 'hidden_threshold_low_dim'

reference_loss = float(best_payload['best_value'])

threshold_results = run_hidden_threshold_grid_progressive(
    data_bundle=data_bundle,
    base_config=base_config,
    hidden_grid=HIDDEN_GRID,
    seeds=SWEEP_SEEDS,
    artifact_root=ARTIFACT_ROOT,
    run_prefix=RUN_PREFIX,
    tolerance_ratio=TOLERANCE_RATIO,
    reference_loss=reference_loss,
    verbose=HIDDEN_VERBOSE,
)

results_df = pd.DataFrame(threshold_results['results_df']).copy()
reference_rows = pd.DataFrame()
if REFERENCE_RESULTS_PATH.exists():
    reference_rows = pd.read_csv(REFERENCE_RESULTS_PATH)
    reference_rows = reference_rows.loc[reference_rows['n_hidden'] == REFERENCE_HIDDEN].copy()
    if not reference_rows.empty:
        if 'device' not in reference_rows.columns:
            reference_rows['device'] = 'mps'
        reference_rows['reference_tag'] = 'reused_h4_reference'

results_with_reference = pd.concat([results_df, reference_rows], ignore_index=True)
if not results_with_reference.empty:
    results_with_reference = (
        results_with_reference
        .sort_values(['n_hidden', 'seed', 'best_val_loss'])
        .drop_duplicates(subset=['n_hidden', 'seed'], keep='first')
        .reset_index(drop=True)
    )

comparison_results_path = ARTIFACT_ROOT / 'final' / f'{RUN_PREFIX}_with_reference_results.csv'
results_with_reference.to_csv(comparison_results_path, index=False)
comparison_figure_path, comparison_threshold_summary = plot_hidden_threshold(
    results_df=results_with_reference,
    save_path=ARTIFACT_ROOT / 'figures' / f'{RUN_PREFIX}_with_reference_threshold.png',
    tolerance_ratio=TOLERANCE_RATIO,
    reference_loss=reference_loss,
    require_plateau=False,
)

comparison_grouped = comparison_threshold_summary['grouped_results'].copy()
baseline_row = comparison_grouped.loc[comparison_grouped['n_hidden'] == REFERENCE_HIDDEN]
baseline_loss = float(baseline_row['mean_val_loss'].iloc[0]) if not baseline_row.empty else np.nan
comparison_grouped['delta_from_prev'] = comparison_grouped['mean_val_loss'].diff()
comparison_grouped['ratio_to_prev'] = comparison_grouped['mean_val_loss'] / comparison_grouped['mean_val_loss'].shift(1)
comparison_grouped['ratio_to_h4'] = (
    comparison_grouped['mean_val_loss'] / baseline_loss if np.isfinite(baseline_loss) else np.nan
)
comparison_grouped_path = ARTIFACT_ROOT / 'final' / f'{RUN_PREFIX}_with_reference_grouped_results.csv'
comparison_grouped.to_csv(comparison_grouped_path, index=False)
comparison_summary_path = ARTIFACT_ROOT / 'final' / f'{RUN_PREFIX}_with_reference_threshold.json'
write_json(
    comparison_summary_path,
    {
        'device': DEVICE,
        'core_type': core_type,
        'hidden_grid': HIDDEN_GRID,
        'reference_hidden': REFERENCE_HIDDEN,
        'reference_results_path': str(REFERENCE_RESULTS_PATH),
        'comparison_results_path': str(comparison_results_path),
        'comparison_grouped_path': str(comparison_grouped_path),
        'comparison_figure_path': str(comparison_figure_path),
        'threshold_limit': comparison_threshold_summary['threshold_limit'],
        'selected_hidden': comparison_threshold_summary['selected_hidden'],
        'threshold_found': comparison_threshold_summary['threshold_found'],
        'acceptable_hidden': comparison_threshold_summary['acceptable_hidden'],
    },
)

selected_hidden = comparison_threshold_summary['selected_hidden']
final_result = None
if RUN_FINAL_RETRAIN and selected_hidden is not None:
    fixed_epochs = int(max(base_config.MIN_EPOCHS, round(float(comparison_grouped.loc[comparison_grouped['n_hidden'] == selected_hidden, 'median_best_epoch'].iloc[0]))))
    final_result = retrain_final_model(
        data_bundle=data_bundle,
        base_config=base_config,
        n_hidden=selected_hidden,
        fixed_epochs=fixed_epochs,
        artifact_root=ARTIFACT_ROOT,
        run_name=FINAL_RUN_NAME,
    )

results_with_reference

In [ ]:
pd.DataFrame(threshold_results['stage_rows'])

In [ ]:
display(comparison_grouped)
display(Image(filename=str(comparison_figure_path)))
comparison_threshold_summary

In [ ]:
final_result